# Notebook for both ARM + dataset code and ARM-only code
Both will produce the same "Top 5 Overall" + "Top 5 Hidden/Secondary" Risk (Died) + Protective (Not Died) tables with:
Analysis_Type | antecedent | consequent | support | support_pct | confidence | confidence_pct | lift

### Code for both generating the summary dataframes needed for the ARM analyses and the ARM analyses themselves (assumes the actual dataset resembles the "dummy"/generated dataset given to me)

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_columns', None)

# ============================================================
# 0) Load + preprocess
# ============================================================
df = pd.read_csv("MDS sample.csv")

df["DATE_ASSESSMENT"] = pd.to_datetime(df["DATE_ASSESSMENT"], format="%d%b%Y", errors="coerce")
df["BENE_DEATH_DT"]   = pd.to_datetime(df["BENE_DEATH_DT"], format="%d%b%Y", errors="coerce")
df["WT_OLD"] = df["WT_OLD"].replace(0, np.nan)

df_sorted = df.sort_values(["BENE_ID", "DATE_ASSESSMENT"], ascending=[True, False]).reset_index(drop=True)

# --- Robust gender/race mapping (only map if it looks coded) ---
def normalize_gender(s):
    s0 = s.copy()
    # if already strings "Male"/"Female", keep
    if s0.dropna().isin(["Male","Female"]).mean() > 0.8:
        return s0
    # map common codes
    return s0.replace({"A":"Male","B":"Female","M":"Male","F":"Female",1:"Male",2:"Female","1":"Male","2":"Female"})

def normalize_race(s):
    s0 = s.copy()
    # if already labeled strings, keep
    if s0.dropna().isin(["White","Black","Hispanic","Asian","American Indian/Alaskan Native",
                         "Native Hawaiian/Pacific Islander","Other","Others","Missing"]).mean() > 0.6:
        return s0
    race_map = {6:"Native Hawaiian/Pacific Islander",5:"American Indian/Alaskan Native",
                4:"Asian",3:"Black",2:"Hispanic",1:"White",0:"Missing"}
    return s0.map(race_map)

df_sorted["GENDER"] = normalize_gender(df_sorted["GENDER"])
df_sorted["RACE"]   = normalize_race(df_sorted["RACE"])

# --- Valid-patient filtering based on earliest assessment ---
first_assessment = (
    df_sorted.sort_values(["BENE_ID","DATE_ASSESSMENT"], ascending=[True, True])
             .drop_duplicates("BENE_ID", keep="first")
             .reset_index(drop=True)
)

mask_age = first_assessment["AGE"] >= 66
mask_wt_notna = first_assessment["WT_OLD"].notna()
low, high = first_assessment["WT_OLD"].quantile([0.01, 0.99])
mask_wt_range = first_assessment["WT_OLD"].between(low, high)
mask_bmi = first_assessment["BMI"] >= 30

valid_patients = first_assessment[mask_age & mask_wt_notna & mask_wt_range & mask_bmi]["BENE_ID"]
df_sorted = df_sorted[df_sorted["BENE_ID"].isin(valid_patients)].copy()

# ============================================================
# 1) Resident-level summary_df
# ============================================================
illness_cols = [
    "ARTHRITIS","STROKE","CANCER","COPD","DIAB","DEMENTIA","DEPRESSION",
    "HEARTFAILURE","HYPERTENSION","ESRD","FT_PROBLEM","HEARING",
    "BOWEL_INCONTINENCE","URINE_INCONTINENCE","ANXTY","MNC_DPRSN","SCHZOPRNIA","PRESSURE_ULCER"
]

def summarize_patient(group):
    first_date = group["DATE_ASSESSMENT"].min()
    last_date  = group["DATE_ASSESSMENT"].max()
    death_date = group["BENE_DEATH_DT"].dropna().min() if group["BENE_DEATH_DT"].notna().any() else pd.NaT

    wt_series = group["WT_OLD"].dropna()
    init_wt = wt_series.iloc[-1] if len(wt_series) else np.nan
    end_wt  = wt_series.iloc[0]  if len(wt_series) else np.nan

    summary = {
        "INITIAL_AGE": group.loc[group["DATE_ASSESSMENT"] == first_date, "AGE"].values[0],
        "BENE_DEATH_DT": death_date if pd.notna(death_date) else np.nan,
        "GENDER": group["GENDER"].dropna().iloc[-1] if group["GENDER"].notna().any() else np.nan,
        "RACE": group["RACE"].dropna().iloc[-1] if group["RACE"].notna().any() else np.nan,
        "ADL_SCORE_INIT": group["ADL_SCORE"].dropna().iloc[-1] if group["ADL_SCORE"].notna().any() else np.nan,
        "Initial_BIMS": group["BIMS"].dropna().iloc[-1] if group["BIMS"].notna().any() else np.nan,
        "Initial_BMI": group["BMI"].dropna().iloc[-1] if group["BMI"].notna().any() else np.nan,
        "Overall_WT_Change_Percent": ((end_wt - init_wt) / init_wt) if pd.notna(init_wt) and init_wt > 0 else np.nan,
    }

    # IMPORTANT: aggregate comorbidities as ANY == 1 if numeric, else fallback to notna()
    for col in illness_cols:
        s = group[col]
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().sum() > 0:
            present = int((s_num.fillna(0).astype(int).clip(0,1) == 1).any())
        else:
            present = int(s.notna().any())
        summary[col] = present

    return pd.Series(summary)

summary_df = df_sorted.groupby("BENE_ID").apply(summarize_patient).reset_index()

# Derived vars
summary_df["ADL_SCORE_INIT"] = pd.to_numeric(summary_df["ADL_SCORE_INIT"], errors="coerce")
summary_df["Initial_BMI"]    = pd.to_numeric(summary_df["Initial_BMI"], errors="coerce")
summary_df["INITIAL_AGE"]    = pd.to_numeric(summary_df["INITIAL_AGE"], errors="coerce")

summary_df["Obesity_Class"] = pd.cut(summary_df["Initial_BMI"], bins=[30,35,40,np.inf],
                                     labels=["Class I","Class II","Class III"])

summary_df["ADL_Impairment"] = pd.cut(summary_df["ADL_SCORE_INIT"], bins=[-np.inf,12,20,28],
                                      labels=["Mild (0-12)","Moderate (13-20)","Severe (21-28)"])

summary_df["died"] = summary_df["BENE_DEATH_DT"].notna().astype(int)

summary_df["Age_Group"] = pd.cut(summary_df["INITIAL_AGE"], bins=[-np.inf,64,74,84,94,np.inf],
                                 labels=["<65","65-74","75-84","85-94","95+"])

# Cognitive_Status (same mapping you’ve been using)
cog_map = {
    "Cognitively intact": "Cognitively intact",
    "Mildly impaired": "Moderate impairment",
    "Moderately impaired": "Moderate impairment",
    "Severely impaired": "Severe impairment"
}
summary_df["Cognitive_Status"] = summary_df["Initial_BIMS"].map(cog_map)

# enforce 0/1 in resident-level comorbidities
summary_df[illness_cols] = summary_df[illness_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int).clip(0,1)

# ============================================================
# 2) Build dfm (complete-case modeling dataset) + illness_cols_final
# ============================================================
required_for_dfm = ["died","Obesity_Class","Age_Group","GENDER","RACE","ADL_Impairment","Cognitive_Status"]
dfm = summary_df.dropna(subset=required_for_dfm).copy()

if dfm.empty:
    raise ValueError(
        "dfm is empty after dropna(). Most likely your GENDER/RACE mapping created NaNs. "
        "Inspect summary_df[['GENDER','RACE']].value_counts(dropna=False)."
    )

illness_cols_final = [c for c in illness_cols if dfm[c].nunique(dropna=False) > 1]

# ============================================================
# 3) ARM transaction matrix
# ============================================================
df_demog_dummies = pd.get_dummies(dfm[["Obesity_Class","ADL_Impairment"]],
                                 prefix=["Obesity_Class","ADL_Impairment"])
X_transaction_master = pd.concat([dfm[illness_cols_final].astype(bool),
                                  df_demog_dummies.astype(bool)], axis=1)

# Quick sanity print: top single supports (should be >0.02 for something)
col_supp = X_transaction_master.mean().sort_values(ascending=False)
print("dfm shape:", dfm.shape)
print("Top supports:\n", col_supp.head(10))
print("Died rate:", dfm["died"].mean())

# Labels
if "illness_labels" not in globals():
    illness_labels = {c: c for c in illness_cols_final}
else:
    for c in illness_cols_final:
        illness_labels.setdefault(c, c)

demog_field_labels = {}
for dummy_col in df_demog_dummies.columns:
    if dummy_col.startswith("Obesity_Class_"):
        demog_field_labels[dummy_col] = f"Obesity: {dummy_col.replace('Obesity_Class_', '')}"
    elif dummy_col.startswith("ADL_Impairment_"):
        demog_field_labels[dummy_col] = f"ADL: {dummy_col.replace('ADL_Impairment_', '')}"

master_feature_labels = {**illness_labels, **demog_field_labels}

def format_pretty_itemset(itemset, outcome_label):
    def clean_name(x):
        if x in ["DIED","NOT_DIED"]:
            return outcome_label
        return master_feature_labels.get(x, x)
    return " + ".join(clean_name(x) for x in sorted(list(itemset)))

# ============================================================
# 4) Robust apriori runner (auto-backs off support if empty)
# ============================================================
MAX_RULE_LEN = 5
MIN_CONF_VAL = 0.20
MIN_SUPPORT_START = 0.02  # try to match your notebook

def apriori_nonempty(X_bool, start_support=0.02, max_len=5):
    supports_to_try = [
        start_support,
        start_support/2,
        start_support/5,
        0.01, 0.005, 0.002, 0.001,
        max(1/len(X_bool), 0.0005)
    ]
    supports_to_try = [s for s in supports_to_try if s > 0]
    for s in supports_to_try:
        freq = apriori(X_bool, min_support=s, use_colnames=True, max_len=max_len)
        if not freq.empty:
            print(f"apriori: using min_support={s:.6f} (freq itemsets={len(freq)})")
            return freq, s
    raise ValueError("No frequent itemsets found even after lowering min_support. Check X matrix.")

# ============================================================
# 5) Top 5 Overall + Hidden/Secondary (deterministic tie-break)
# ============================================================
def build_top5_and_hidden(rules_df, analysis_name_overall, analysis_name_hidden, top_k=5, n_exclude=3):
    rules_sorted = (
        rules_df[["antecedent","consequent","support","support_pct","confidence","confidence_pct","lift"]]
        .sort_values(["lift","confidence","support","antecedent"], ascending=[False, False, False, True])
        .reset_index(drop=True)
    )
    top_overall = rules_sorted.head(top_k).copy()
    top_overall.insert(0, "Analysis_Type", analysis_name_overall)

    antecedent_items = []
    for itemset in rules_df["antecedents"]:
        antecedent_items.extend(list(itemset))

    top_confounders = [item for item, _ in Counter(antecedent_items).most_common(n_exclude)]
    readable = [master_feature_labels.get(x, x) for x in top_confounders]
    print(f"--> [{analysis_name_overall}] Confounders removed for Hidden/Secondary: {readable}")

    rules_hidden = rules_df[rules_df["antecedents"].apply(lambda s: not any(x in s for x in top_confounders))].copy()

    hidden_sorted = (
        rules_hidden[["antecedent","consequent","support","support_pct","confidence","confidence_pct","lift"]]
        .sort_values(["lift","confidence","support","antecedent"], ascending=[False, False, False, True])
        .head(top_k)
        .reset_index(drop=True)
    )
    hidden_sorted.insert(0, "Analysis_Type", analysis_name_hidden)

    return pd.concat([top_overall, hidden_sorted], ignore_index=True)

# ============================================================
# 6) DIED rules
# ============================================================
X_died = X_transaction_master.copy()
X_died["DIED"] = dfm["died"].astype(bool)

freq_died, used_support_died = apriori_nonempty(X_died, start_support=MIN_SUPPORT_START, max_len=MAX_RULE_LEN)
rules_died = association_rules(freq_died, metric="confidence", min_threshold=MIN_CONF_VAL)

rules_died = rules_died[rules_died["consequents"].apply(lambda s: (len(s)==1 and "DIED" in s))].copy()

rules_died["antecedent"] = rules_died["antecedents"].apply(lambda s: format_pretty_itemset(s, "Died"))
rules_died["consequent"] = rules_died["consequents"].apply(lambda s: format_pretty_itemset(s, "Died"))
rules_died["support_pct"] = (rules_died["support"] * 100).round(2)
rules_died["confidence_pct"] = (rules_died["confidence"] * 100).round(2)
rules_died["lift"] = rules_died["lift"].round(3)

comprehensive_died_report = build_top5_and_hidden(
    rules_died,
    "Top 5 Overall Risk Profiles",
    "Top 5 Hidden/Secondary Risk Profiles",
    top_k=5,
    n_exclude=3
)

print("\n======================================== MORTALITY PROFILES (DIED) ========================================")
display(comprehensive_died_report)

# ============================================================
# 7) NOT DIED rules
# ============================================================
X_not_died = X_transaction_master.copy()
X_not_died["NOT_DIED"] = ~dfm["died"].astype(bool)

freq_not_died, used_support_not = apriori_nonempty(X_not_died, start_support=MIN_SUPPORT_START, max_len=MAX_RULE_LEN)
rules_not_died = association_rules(freq_not_died, metric="confidence", min_threshold=MIN_CONF_VAL)

rules_not_died = rules_not_died[rules_not_died["consequents"].apply(lambda s: (len(s)==1 and "NOT_DIED" in s))].copy()

rules_not_died["antecedent"] = rules_not_died["antecedents"].apply(lambda s: format_pretty_itemset(s, "Not Died"))
rules_not_died["consequent"] = rules_not_died["consequents"].apply(lambda s: format_pretty_itemset(s, "Not Died"))
rules_not_died["support_pct"] = (rules_not_died["support"] * 100).round(2)
rules_not_died["confidence_pct"] = (rules_not_died["confidence"] * 100).round(2)
rules_not_died["lift"] = rules_not_died["lift"].round(3)

comprehensive_not_died_report = build_top5_and_hidden(
    rules_not_died,
    "Top 5 Overall Protective Profiles",
    "Top 5 Hidden/Secondary Protective Profiles",
    top_k=5,
    n_exclude=3
)

print("\n======================================== SURVIVAL PROFILES (NOT DIED) ========================================")
display(comprehensive_not_died_report)

/var/folders/0y/mmp_lnms4tx1tvdz0cwsqdx40000gs/T/ipykernel_23399/1441690965.py:11: DtypeWarning: Columns (0: GG0130E1_BTHE_SELF_STRT_CD, 1: GG0130E3_BTHE_SELF_END_CD, 2: GG0130F1_UPR_DRSNG_STRT_CD, 3: GG0130F3_UPR_DRSNG_END_CD, 4: GG0130G1_LWR_DRSNG_STRT_CD, 5: GG0130G3_LWR_DRSNG_END_CD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("MDS sample.csv")


dfm shape: (417, 32)
Top supports:
 FT_PROBLEM                         0.995204
URINE_INCONTINENCE                 0.932854
HYPERTENSION                       0.918465
BOWEL_INCONTINENCE                 0.877698
DEPRESSION                         0.685851
HEARTFAILURE                       0.611511
DIAB                               0.577938
ADL_Impairment_Moderate (13-20)    0.546763
Obesity_Class_Class I              0.539568
DEMENTIA                           0.525180
dtype: float64
Died rate: 0.4940047961630695
apriori: using min_support=0.020000 (freq itemsets=16749)
--> [Top 5 Overall Risk Profiles] Confounders removed for Hidden/Secondary: ['FT_PROBLEM', 'URINE_INCONTINENCE', 'HYPERTENSION']

======================================== MORTALITY PROFILES (DIED) ========================================


,Analysis_Type,antecedent,consequent,support,support_pct,confidence,confidence_pct,lift
0,Top 5 Overall Risk Profiles,ANXTY + BOWEL_INCONTINENCE + CANCER + COPD,Died,0.023981,2.40,1.000000,100.00,2.024
1,Top 5 Overall Risk Profiles,ADL: Mild (0-12) + CANCER + FT_PROBLEM + HEART...,Died,0.021583,2.16,1.000000,100.00,2.024
2,Top 5 Overall Risk Profiles,ADL: Mild (0-12) + CANCER + HEARTFAILURE,Died,0.021583,2.16,1.000000,100.00,2.024
3,Top 5 Overall Risk Profiles,ADL: Mild (0-12) + CANCER + HEARTFAILURE + HYP...,Died,0.021583,2.16,1.000000,100.00,2.024
4,Top 5 Overall Risk Profiles,ADL: Moderate (13-20) + ESRD + HEARTFAILURE + ...,Died,0.026379,2.64,0.916667,91.67,1.856
5,Top 5 Hidden/Secondary Risk Profiles,ANXTY + BOWEL_INCONTINENCE + CANCER + COPD,Died,0.023981,2.40,1.000000,100.00,2.024
6,Top 5 Hidden/Secondary Risk Profiles,ADL: Mild (0-12) + CANCER + HEARTFAILURE,Died,0.021583,2.16,1.000000,100.00,2.024
7,Top 5 Hidden/Secondary Risk Profiles,ADL: Moderate (13-20) + ESRD + HEARTFAILURE + ...,Died,0.026379,2.64,0.916667,91.67,1.856
8,Top 5 Hidden/Secondary Risk Profiles,ADL: Moderate (13-20) + BOWEL_INCONTINENCE + C...,Died,0.023981,2.40,0.909091,90.91,1.840
9,Top 5 Hidden/Secondary Risk Profiles,ADL: Severe (21-28) + ANXTY + COPD + DIAB,Died,0.023981,2.40,0.909091,90.91,1.840


apriori: using min_support=0.020000 (freq itemsets=16272)
--> [Top 5 Overall Protective Profiles] Confounders removed for Hidden/Secondary: ['FT_PROBLEM', 'URINE_INCONTINENCE', 'HYPERTENSION']

======================================== SURVIVAL PROFILES (NOT DIED) ========================================


,Analysis_Type,antecedent,consequent,support,support_pct,confidence,confidence_pct,lift
0,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.031175,3.12,0.866667,86.67,1.713
1,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + FT_PROBLEM + MNC_DPRSN,Not Died,0.026379,2.64,0.846154,84.62,1.672
2,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + HYPERTENSION + MNC_...,Not Died,0.026379,2.64,0.846154,84.62,1.672
3,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + MNC_DPRSN,Not Died,0.026379,2.64,0.846154,84.62,1.672
4,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.021583,2.16,0.818182,81.82,1.617
5,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.031175,3.12,0.866667,86.67,1.713
6,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + MNC_DPRSN,Not Died,0.026379,2.64,0.846154,84.62,1.672
7,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.021583,2.16,0.818182,81.82,1.617
8,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + DEMENTIA + HEARTFAI...,Not Died,0.021583,2.16,0.818182,81.82,1.617
9,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + DEPRESSION + MNC_DPRSN,Not Died,0.021583,2.16,0.818182,81.82,1.617


### Code for just the ARM analyses, will require renaming either the dataset itself or some of the variable names in the code at the minimum

In [3]:
import numpy as np
import pandas as pd
from collections import Counter
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_columns", None)

# ------------------------------------------------------------
# USER SETTINGS (match your notebook defaults)
# ------------------------------------------------------------
MIN_SUPPORT_VAL = 0.02
MIN_CONF_VAL    = 0.20
MAX_RULE_LEN    = 5
TOP_K_RULES     = 5
N_EXCLUDE       = 3   # dominant antecedent items to remove for Hidden/Secondary

# ------------------------------------------------------------
# 1) Choose the dataset to mine (to match final_code behavior)
#    Prefer dfm (non-null modeling dataset) if it exists.
# ------------------------------------------------------------
if 'dfm' in globals():
    df_arm = dfm.copy()
elif 'summary_df2' in globals():
    df_arm = summary_df2.copy()
elif 'summary_df' in globals():
    df_arm = summary_df.copy()
else:
    raise ValueError("Need dfm or summary_df2 or summary_df in memory before running ARM.")

# ------------------------------------------------------------
# 2) Ensure required fields exist: died, Obesity_Class, ADL_Impairment
# ------------------------------------------------------------
# died
if 'died' not in df_arm.columns:
    if 'BENE_DEATH_DT' in df_arm.columns:
        df_arm['died'] = df_arm['BENE_DEATH_DT'].notna().astype(int)
    else:
        raise ValueError("No 'died' or 'BENE_DEATH_DT' column found.")

# Obesity_Class (if missing)
if 'Obesity_Class' not in df_arm.columns:
    if 'Initial_BMI' not in df_arm.columns:
        raise ValueError("Missing 'Obesity_Class' and 'Initial_BMI' needed to create it.")
    df_arm['Initial_BMI'] = pd.to_numeric(df_arm['Initial_BMI'], errors='coerce')
    df_arm['Obesity_Class'] = pd.cut(
        df_arm['Initial_BMI'],
        bins=[30, 35, 40, np.inf],
        labels=['Class I', 'Class II', 'Class III']
    )

# ADL_Impairment (if missing)
if 'ADL_Impairment' not in df_arm.columns:
    if 'ADL_SCORE_INIT' not in df_arm.columns:
        raise ValueError("Missing 'ADL_Impairment' and 'ADL_SCORE_INIT' needed to create it.")
    df_arm['ADL_SCORE_INIT'] = pd.to_numeric(df_arm['ADL_SCORE_INIT'], errors='coerce')
    df_arm['ADL_Impairment'] = pd.cut(
        df_arm['ADL_SCORE_INIT'],
        bins=[-np.inf, 12, 20, 28],
        labels=['Mild (0-12)', 'Moderate (13-20)', 'Severe (21-28)']
    )

# ------------------------------------------------------------
# 3) Choose comorbidity/condition columns (match final_code logic)
#    Prefer illness_cols_final if it exists; else illness_cols; else fallback list.
# ------------------------------------------------------------
if 'illness_cols_final' in globals():
    cond_cols = list(illness_cols_final)
elif 'illness_cols' in globals():
    cond_cols = list(illness_cols)
else:
    cond_cols = [
        'ARTHRITIS','STROKE','CANCER','COPD','DIAB','DEMENTIA','DEPRESSION',
        'HEARTFAILURE','HYPERTENSION','ESRD','FT_PROBLEM','HEARING',
        'BOWEL_INCONTINENCE','URINE_INCONTINENCE','ANXTY','MNC_DPRSN','SCHZOPRNIA','PRESSURE_ULCER'
    ]

# Ensure condition columns exist
missing = [c for c in cond_cols if c not in df_arm.columns]
if missing:
    raise ValueError(f"Missing condition columns in df_arm: {missing}")

# Force 0/1
df_arm[cond_cols] = (
    df_arm[cond_cols]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
    .clip(0, 1)
)

# ------------------------------------------------------------
# 4) Restrict to rows with the covariates used by ARM (matches dfm behavior)
#    and drop constant condition columns if illness_cols_final wasn't supplied.
# ------------------------------------------------------------
df_arm = df_arm.dropna(subset=['Obesity_Class', 'ADL_Impairment']).copy()

if 'illness_cols_final' not in globals():
    nunq = df_arm[cond_cols].nunique(dropna=False)
    cond_cols = [c for c in cond_cols if nunq[c] > 1]  # drop all-0 or all-1 columns

# ------------------------------------------------------------
# 5) Build transaction matrix: diseases + ADL dummies + obesity dummies
# ------------------------------------------------------------
categorical_covariates = ['Obesity_Class', 'ADL_Impairment']
df_demog_dummies = pd.get_dummies(df_arm[categorical_covariates], prefix=categorical_covariates)

X_diseases = df_arm[cond_cols]
X_master = pd.concat([X_diseases, df_demog_dummies], axis=1).astype(bool)

# ------------------------------------------------------------
# 6) Pretty labels (use your notebook's illness_labels if present)
# ------------------------------------------------------------
if 'illness_labels' not in globals():
    illness_labels = {c: c for c in cond_cols}

demog_field_labels = {}
for dummy_col in df_demog_dummies.columns:
    if dummy_col.startswith('Obesity_Class_'):
        demog_field_labels[dummy_col] = f"Obesity: {dummy_col.replace('Obesity_Class_', '')}"
    elif dummy_col.startswith('ADL_Impairment_'):
        demog_field_labels[dummy_col] = f"ADL: {dummy_col.replace('ADL_Impairment_', '')}"

master_feature_labels = {**illness_labels, **demog_field_labels}

def format_pretty_itemset(itemset, outcome_label):
    def clean_name(x):
        if x in ['DIED', 'NOT_DIED']:
            return outcome_label
        return master_feature_labels.get(x, x)
    return " + ".join(clean_name(x) for x in sorted(list(itemset)))

# ------------------------------------------------------------
# 7) Helper: Build "Top 5 Overall" and "Top 5 Hidden/Secondary" (deterministic ties)
# ------------------------------------------------------------
def build_top5_and_hidden(rules_df, analysis_name_overall, analysis_name_hidden,
                          top_k=TOP_K_RULES, n_exclude=N_EXCLUDE):

    # Deterministic sort key: lift, confidence, support, antecedent (string)
    rules_sorted = (
        rules_df[['antecedent','consequent','support','support_pct','confidence','confidence_pct','lift','antecedents']]
        .sort_values(['lift','confidence','support','antecedent'], ascending=[False, False, False, True])
        .reset_index(drop=True)
    )

    top_overall = rules_sorted.head(top_k).copy()
    top_overall.insert(0, 'Analysis_Type', analysis_name_overall)
    top_overall = top_overall.drop(columns=['antecedents'])

    # Find most common raw antecedent items (from frozensets)
    antecedent_items = []
    for aset in rules_df['antecedents']:
        antecedent_items.extend(list(aset))

    top_confounders = [item for item, _ in Counter(antecedent_items).most_common(n_exclude)]
    readable = [master_feature_labels.get(x, x) for x in top_confounders]
    print(f"--> [{analysis_name_overall}] Confounders removed for Hidden/Secondary: {readable}")

    rules_hidden = rules_df[
        rules_df['antecedents'].apply(lambda s: not any(x in s for x in top_confounders))
    ].copy()

    hidden_sorted = (
        rules_hidden[['antecedent','consequent','support','support_pct','confidence','confidence_pct','lift','antecedents']]
        .sort_values(['lift','confidence','support','antecedent'], ascending=[False, False, False, True])
        .head(top_k)
        .reset_index(drop=True)
    )
    hidden_sorted.insert(0, 'Analysis_Type', analysis_name_hidden)
    hidden_sorted = hidden_sorted.drop(columns=['antecedents'])

    return pd.concat([top_overall, hidden_sorted], ignore_index=True)

# ------------------------------------------------------------
# 8) Run ARM for DIED and NOT_DIED in the exact output format you want
# ------------------------------------------------------------
def run_arm_outcome(outcome_colname, outcome_mask, outcome_label,
                    overall_label, hidden_label):

    X = X_master.copy()
    X[outcome_colname] = outcome_mask.astype(bool)

    freq = apriori(X, min_support=MIN_SUPPORT_VAL, use_colnames=True, max_len=MAX_RULE_LEN)
    rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONF_VAL)

    # Keep only rules with consequent == {outcome_colname}
    rules = rules[rules['consequents'].apply(lambda s: (len(s)==1 and outcome_colname in s))].copy()

    # Pretty strings + metrics
    rules['antecedent'] = rules['antecedents'].apply(lambda s: format_pretty_itemset(s, outcome_label))
    rules['consequent'] = rules['consequents'].apply(lambda s: format_pretty_itemset(s, outcome_label))
    rules['support_pct'] = (rules['support'] * 100).round(2)
    rules['confidence_pct'] = (rules['confidence'] * 100).round(2)
    rules['lift'] = rules['lift'].round(3)

    # Select + output tables
    report = build_top5_and_hidden(
        rules,
        analysis_name_overall=overall_label,
        analysis_name_hidden=hidden_label
    )

    # Keep the exact columns ordering you showed
    report = report[['Analysis_Type','antecedent','consequent','support','support_pct','confidence','confidence_pct','lift']]
    return report

# Risk profiles: DIED
risk_report = run_arm_outcome(
    outcome_colname='DIED',
    outcome_mask=df_arm['died'].astype(int) == 1,
    outcome_label='Died',
    overall_label='Top 5 Overall Risk Profiles',
    hidden_label='Top 5 Hidden/Secondary Risk Profiles'
)

print("\n======================================== MORTALITY PROFILES (DIED) ========================================")
display(risk_report)

# Protective profiles: NOT_DIED
protect_report = run_arm_outcome(
    outcome_colname='NOT_DIED',
    outcome_mask=df_arm['died'].astype(int) == 0,
    outcome_label='Not Died',
    overall_label='Top 5 Overall Protective Profiles',
    hidden_label='Top 5 Hidden/Secondary Protective Profiles'
)

print("\n======================================== SURVIVAL PROFILES (NOT DIED) ========================================")
display(protect_report)

# Optional: save
# risk_report.to_csv("arm_top5_risk_profiles.csv", index=False)
# protect_report.to_csv("arm_top5_protective_profiles.csv", index=False)

--> [Top 5 Overall Risk Profiles] Confounders removed for Hidden/Secondary: ['FT_PROBLEM', 'URINE_INCONTINENCE', 'HYPERTENSION']

======================================== MORTALITY PROFILES (DIED) ========================================


,Analysis_Type,antecedent,consequent,support,support_pct,confidence,confidence_pct,lift
0,Top 5 Overall Risk Profiles,ANXTY + BOWEL_INCONTINENCE + CANCER + COPD,Died,0.023981,2.40,1.000000,100.00,2.024
1,Top 5 Overall Risk Profiles,ADL: Mild (0-12) + CANCER + FT_PROBLEM + HEART...,Died,0.021583,2.16,1.000000,100.00,2.024
2,Top 5 Overall Risk Profiles,ADL: Mild (0-12) + CANCER + HEARTFAILURE,Died,0.021583,2.16,1.000000,100.00,2.024
3,Top 5 Overall Risk Profiles,ADL: Mild (0-12) + CANCER + HEARTFAILURE + HYP...,Died,0.021583,2.16,1.000000,100.00,2.024
4,Top 5 Overall Risk Profiles,ADL: Moderate (13-20) + ESRD + HEARTFAILURE + ...,Died,0.026379,2.64,0.916667,91.67,1.856
5,Top 5 Hidden/Secondary Risk Profiles,ANXTY + BOWEL_INCONTINENCE + CANCER + COPD,Died,0.023981,2.40,1.000000,100.00,2.024
6,Top 5 Hidden/Secondary Risk Profiles,ADL: Mild (0-12) + CANCER + HEARTFAILURE,Died,0.021583,2.16,1.000000,100.00,2.024
7,Top 5 Hidden/Secondary Risk Profiles,ADL: Moderate (13-20) + ESRD + HEARTFAILURE + ...,Died,0.026379,2.64,0.916667,91.67,1.856
8,Top 5 Hidden/Secondary Risk Profiles,ADL: Moderate (13-20) + BOWEL_INCONTINENCE + C...,Died,0.023981,2.40,0.909091,90.91,1.840
9,Top 5 Hidden/Secondary Risk Profiles,ADL: Severe (21-28) + ANXTY + COPD + DIAB,Died,0.023981,2.40,0.909091,90.91,1.840


--> [Top 5 Overall Protective Profiles] Confounders removed for Hidden/Secondary: ['FT_PROBLEM', 'URINE_INCONTINENCE', 'HYPERTENSION']

======================================== SURVIVAL PROFILES (NOT DIED) ========================================


,Analysis_Type,antecedent,consequent,support,support_pct,confidence,confidence_pct,lift
0,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.031175,3.12,0.866667,86.67,1.713
1,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + FT_PROBLEM + MNC_DPRSN,Not Died,0.026379,2.64,0.846154,84.62,1.672
2,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + HYPERTENSION + MNC_...,Not Died,0.026379,2.64,0.846154,84.62,1.672
3,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + MNC_DPRSN,Not Died,0.026379,2.64,0.846154,84.62,1.672
4,Top 5 Overall Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.021583,2.16,0.818182,81.82,1.617
5,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.031175,3.12,0.866667,86.67,1.713
6,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + MNC_DPRSN,Not Died,0.026379,2.64,0.846154,84.62,1.672
7,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + BOWEL_INCONTINENCE ...,Not Died,0.021583,2.16,0.818182,81.82,1.617
8,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + DEMENTIA + HEARTFAI...,Not Died,0.021583,2.16,0.818182,81.82,1.617
9,Top 5 Hidden/Secondary Protective Profiles,ADL: Mild (0-12) + ANXTY + DEPRESSION + MNC_DPRSN,Not Died,0.021583,2.16,0.818182,81.82,1.617
